In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE server_daily_summary (
    server_id TEXT,
    collection_date TEXT,
    avg_cpu REAL,
    region TEXT
);
INSERT INTO server_daily_summary VALUES
    ('srv-01','2026-02-01',72.5,'us-east'),
    ('srv-01','2026-02-02',74.1,'us-east'),
    ('srv-01','2026-02-03',68.9,'us-east'),
    ('srv-01','2026-02-04',79.3,'us-east'),
    ('srv-01','2026-02-05',85.2,'us-east'),
    ('srv-01','2026-02-06',71.0,'us-east'),
    ('srv-01','2026-02-07',88.4,'us-east'),
    ('srv-02','2026-02-01',45.2,'us-east'),
    ('srv-02','2026-02-02',48.1,'us-east'),
    ('srv-02','2026-02-03',51.3,'us-east'),
    ('srv-02','2026-02-04',47.9,'us-east'),
    ('srv-02','2026-02-05',44.0,'us-east'),
    ('srv-02','2026-02-06',52.7,'us-east'),
    ('srv-02','2026-02-07',49.8,'us-east'),
    ('srv-03','2026-02-01',91.3,'us-west'),
    ('srv-03','2026-02-02',88.7,'us-west'),
    ('srv-03','2026-02-03',92.1,'us-west'),
    ('srv-03','2026-02-04',89.5,'us-west'),
    ('srv-03','2026-02-05',95.0,'us-west'),
    ('srv-03','2026-02-06',90.3,'us-west'),
    ('srv-03','2026-02-07',93.8,'us-west'),
    ('srv-04','2026-02-01',33.1,'us-west'),
    ('srv-04','2026-02-02',35.5,'us-west'),
    ('srv-04','2026-02-03',31.8,'us-west'),
    ('srv-04','2026-02-04',38.2,'us-west'),
    ('srv-04','2026-02-05',36.9,'us-west'),
    ('srv-04','2026-02-06',34.4,'us-west'),
    ('srv-04','2026-02-07',37.1,'us-west');
""")
print("Sample database ready!")
print("Table: server_daily_summary (28 rows, 4 servers, 7 days)")
print()
print(pd.read_sql_query("SELECT * FROM server_daily_summary LIMIT 6", conn).to_string(index=False))

# SQL — Advanced Window Functions
**Day 2 — SQL Module**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>Core Insight:</strong> You know ROW_NUMBER, RANK, LAG/LEAD, and rolling aggregates.
What separates senior candidates: NTILE, PERCENT_RANK, FIRST_VALUE/LAST_VALUE,
and named WINDOW clauses. These show up in business analytics — quartile ranking,
percentile scoring, baseline comparison, and DRY SQL.
</div>

**Reminder:** Window functions keep all rows. GROUP BY collapses rows. This is the fundamental difference.

## NTILE — Quartile / Bucket Analysis

`NTILE(n)` divides the result set into n equally-sized buckets, numbered 1 to n.
- Bucket 1 = highest (if ORDER BY DESC)
- Most common use: quartile analysis (n=4), decile analysis (n=10)
- Handles uneven divisions: earlier buckets get +1 row when n doesn't divide evenly

In [ ]:
# NTILE: divide servers into 4 quartiles by average CPU (using first day only for clarity)
sql_ntile = """
SELECT
    server_id,
    ROUND(AVG(avg_cpu), 2) AS avg_cpu_overall,
    NTILE(4) OVER (ORDER BY AVG(avg_cpu) DESC) AS quartile
FROM server_daily_summary
GROUP BY server_id
ORDER BY avg_cpu_overall DESC
"""

result = pd.read_sql_query(sql_ntile, conn)
print("Server CPU Quartile Analysis (Q1=highest, Q4=lowest):")
print(result.to_string(index=False))
print()
print("NTILE(4): divides rows into 4 equal buckets. With 4 servers, each gets 1.")
print("With 10 servers and NTILE(4): buckets would be 3,3,2,2 rows each.")

## PERCENT_RANK and CUME_DIST

**PERCENT_RANK:** `(rank - 1) / (total_rows - 1)` — always 0 for first row, 1 for last
**CUME_DIST:** `rows_leq_current / total_rows` — fraction of rows ≤ current value

> **Note:** PERCENT_RANK and CUME_DIST require SQLite 3.44.0+ (Nov 2023).
> The cell below uses pandas to simulate these for maximum compatibility.

In [ ]:
# PERCENT_RANK and CUME_DIST simulation using pandas
# (equivalent to what you would write in production SQL)

df = pd.read_sql_query("""
    SELECT server_id, ROUND(AVG(avg_cpu),2) AS avg_cpu
    FROM server_daily_summary GROUP BY server_id ORDER BY avg_cpu
""", conn)

n = len(df)
df = df.sort_values('avg_cpu').reset_index(drop=True)
df['rank'] = df['avg_cpu'].rank(method='min').astype(int)
df['percent_rank'] = ((df['rank'] - 1) / (n - 1)).round(3)
df['cume_dist'] = (df['avg_cpu'].rank(method='max') / n).round(3)

print("PERCENT_RANK and CUME_DIST simulation:")
print(df.to_string(index=False))
print()
print("PERCENT_RANK: (rank-1)/(n-1) — first=0, last=1")
print("CUME_DIST:    rows<=current/n — always > 0")
print()
print("In SQL (requires SQLite 3.44+ or PostgreSQL/Snowflake/DuckDB):")
print("""
SELECT server_id, avg_cpu,
    ROUND(PERCENT_RANK() OVER (ORDER BY avg_cpu) * 100, 1) AS percentile,
    ROUND(CUME_DIST()    OVER (ORDER BY avg_cpu) * 100, 1) AS cumulative_pct
FROM server_daily_summary;
""")

## FIRST_VALUE / LAST_VALUE — Baseline Comparison

Compare each row to the first (or last) value in its partition:
- `FIRST_VALUE(col) OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` — first value in partition
- **LAST_VALUE gotcha:** Default frame is up to current row. You must explicitly set `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` to get the true last value.

In [ ]:
# FIRST_VALUE: compare each day's CPU to the first day of the week (baseline)
sql_first = """
SELECT
    server_id,
    collection_date,
    ROUND(avg_cpu, 2) AS avg_cpu,
    ROUND(FIRST_VALUE(avg_cpu) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS baseline_cpu,
    ROUND(avg_cpu - FIRST_VALUE(avg_cpu) OVER (
        PARTITION BY server_id
        ORDER BY collection_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS delta_from_baseline
FROM server_daily_summary
WHERE server_id IN ('srv-01','srv-03')
ORDER BY server_id, collection_date
"""

result = pd.read_sql_query(sql_first, conn)
print("FIRST_VALUE: CPU delta from each server's day-1 baseline:")
print(result.to_string(index=False))
print()
print("Positive delta_from_baseline = CPU trending up vs baseline.")
print("srv-03 shows consistently high usage — capacity risk.")

## Named WINDOW Clause — DRY SQL

Define the window once, reuse it across multiple calculations. Prevents copy-paste errors.

```sql
SELECT
    server_id, collection_date, avg_cpu,
    AVG(avg_cpu) OVER w AS rolling_7d_avg,
    MAX(avg_cpu) OVER w AS rolling_7d_max,
    MIN(avg_cpu) OVER w AS rolling_7d_min
FROM server_daily_summary
WINDOW w AS (
    PARTITION BY server_id
    ORDER BY collection_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
);
```

The `WINDOW` clause requires SQLite 3.28.0+. Supported in PostgreSQL, Snowflake, BigQuery, DuckDB.

In [ ]:
# Named WINDOW clause: rolling 3-day stats per server
# Uses WINDOW w AS (...) to define once, reference multiple times
sql_window = """
SELECT
    server_id,
    collection_date,
    ROUND(avg_cpu, 2) AS avg_cpu,
    ROUND(AVG(avg_cpu) OVER w, 2) AS roll_3d_avg,
    ROUND(MAX(avg_cpu) OVER w, 2) AS roll_3d_max,
    ROUND(MIN(avg_cpu) OVER w, 2) AS roll_3d_min
FROM server_daily_summary
WHERE server_id = 'srv-01'
WINDOW w AS (
    PARTITION BY server_id
    ORDER BY collection_date
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
)
ORDER BY collection_date
"""

result = pd.read_sql_query(sql_window, conn)
print("Named WINDOW clause: rolling 3-day stats for srv-01:")
print(result.to_string(index=False))
print()
print("WINDOW w defined once, referenced 3 times (AVG, MAX, MIN).")
print("Without WINDOW clause: copy the full OVER() 3 times — error-prone.")

## Interview Q&A

**Q: What is NTILE(4) useful for in capacity planning?**
A: Quartile analysis — "which servers are in the top 25% of CPU usage?" Q1 = at-risk tier, Q2 = monitor tier. Standard in executive capacity dashboards.

**Q: PERCENT_RANK vs CUME_DIST — what's the key difference?**
A: PERCENT_RANK = (rank-1)/(n-1) — first row is always 0. CUME_DIST = rows_leq_current/n — always > 0. PERCENT_RANK is position-based; CUME_DIST includes ties differently.

**Q: Why use the WINDOW clause?**
A: DRY principle — define the window once, use it in multiple expressions. One change (e.g., change ROWS from 6 to 13 PRECEDING) updates all calculations. Prevents drift bugs.

**Q: LAST_VALUE gotcha?**
A: Default frame is ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW. So LAST_VALUE returns the *current row's value*, not the last in the partition. Fix: use `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING`.

**Q: Window functions vs GROUP BY — when do you use each?**
A: GROUP BY when you want one row per group (aggregation). Window functions when you want to add group statistics to each individual row while keeping all rows.

## The Citi Angle

**At Citi, these patterns were daily work:**

1. **NTILE** — Monthly capacity tiering: "assign each server to a risk tier (Q1=critical, Q2=high, Q3=medium, Q4=low) based on 30-day average CPU"

2. **FIRST_VALUE** — Trend analysis: "how much has this server's CPU grown since the start of the month?" Delta from baseline is the key input to capacity forecasting.

3. **Named WINDOW** — Complex dashboard queries with 5-6 window functions over the same partition needed the WINDOW clause to stay maintainable.

**The behavioral story:**
*"At Citi, I rewrote the capacity monitoring query from self-joins to window functions — the NTILE + rolling average approach. Query time dropped from ~8 seconds to under 1 second on 500M rows. The named WINDOW clause meant the business could adjust the rolling period in one place and all metrics updated consistently."*